# Walkthrough: `build_lineage.py`

This notebook reconstructs the `build_lineage.py` script in this directory
**block by block** so you can see how every piece works, run them one at a
time, and modify the functions as you go.

It uses the flat `istari_digital_client.Client` (v2) for the lineage walk
and `V3Client` for the optional report upload — no fluent helpers.

By the end you'll have:

- A `LineageNode` tree rooted at any revision you choose
- An ASCII tree print of that tree
- A self-contained Markdown report (rendered inline)
- (Optionally) the same report uploaded as a v3 `Model` + child `Artifact`
  joined by a `produces` relationship, plus a new v2 `SystemConfiguration`
  on a system of your choice

## Drift note

This notebook reconstructs `build_lineage.py`. When that script changes —
new helpers, signature tweaks, behaviour fixes — re-run the notebook and
update the affected cells. The two files are intended to stay in sync but
there is no automated check: it's reviewer responsibility on any PR that
touches either.

## Prerequisites

- The cookbook's `istari-labs-helpers` uv environment is activated *or*
  you've installed `istari-digital-client` and `python-dotenv` directly.
- An Istari registry URL and a personal access token are exported as env
  vars or live in a `.env` file in the notebook's working directory.
- Optionally, a few known UUIDs from your environment (a system, a
  revision, etc.). See the **Parameters** cell below.


## 1 · Setup

Imports. The walk uses `Client` (v2) for revisions/jobs/system metadata
and `V3Client` for the upload-time `create_resource` /
`create_revision_relationship` calls.


In [ ]:
import json
import os
import sys
import tempfile
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from istari_digital_client import Client, Configuration, V3Client
from istari_digital_client.v2.models import (
    FileRevision,
    NewSystemConfiguration,
    NewTrackedFile,
    TrackedFile,
    TrackedFileSpecifierType,
)
from istari_digital_client.v3.models import (
    NewRevisionRelationshipDto,
    ResourceTypeDto,
)

## 2 · Parameters

Set whichever of these you want to demo. Empty values cause the relevant
section to print a skip message instead of erroring — so you can run the
notebook top-to-bottom even if you've only filled in one or two.

| Constant | Used by | How to obtain |
|---|---|---|
| `SYSTEM_ID` | "walk a whole system" sections | URL of any system in the Istari UI (`/systems/<id>`) |
| `REVISION_ID` | "trace a single revision" sections | Output of `run_chain.py` (`final_named_cells_revision_id`) or any `rev.id` you have |
| `RESOURCE_ID` | alternative to `REVISION_ID` | Any resource UUID; we resolve its current revision via v3 |
| `USER_ID` | optional `UserCache` demo | Any user UUID on the registry |
| `JOB_ID` | optional `_job_metadata` demo | Output of `run_chain.py` (`job1_id` / `job2_id`) |

`RUN_UPLOAD` gates every side-effectful step. Leave it `False` while
reading; flip it to `True` once you actually want to create resources on
the platform.


In [ ]:
# --- Required for the "walk a system" sections ---
SYSTEM_ID = ""

# --- Required for the "trace a single revision" sections ---
# Provide ONE of these. RESOURCE_ID is resolved to its current revision via v3.
REVISION_ID = ""
RESOURCE_ID = ""

# --- Optional demos ---
USER_ID = ""   # any user uuid you know — used to demo UserCache
JOB_ID  = ""   # any job uuid    — used to demo _job_metadata

# --- Side-effects gate ---
RUN_UPLOAD = False   # flip to True to actually upload the report

# --- Walk depth ---
MAX_DEPTH = 12

## 3 · Connect to the registry

`Configuration` reads `ISTARI_REGISTRY_URL` and a token from the env. The
v2 `Client` and v3 `V3Client` both share the same `Configuration` instance
so they hit the same registry with the same credentials.

`readiness_check()` confirms the token works before we do anything that
costs real round-trips.


In [ ]:
load_dotenv(override=False)
registry_url = os.getenv("ISTARI_REGISTRY_URL")
token = os.getenv("ISTARI_PERSONAL_ACCESS_TOKEN") or os.getenv("ISTARI_REGISTRY_AUTH_TOKEN")
if not registry_url or not token:
    raise RuntimeError(
        "Set ISTARI_REGISTRY_URL and one of "
        "ISTARI_PERSONAL_ACCESS_TOKEN / ISTARI_REGISTRY_AUTH_TOKEN"
    )

config = Configuration(registry_url=registry_url, registry_auth_token=token)
client = Client(config)
v3 = V3Client(config)

health = client.readiness_check()
assert health.healthy, f"Platform unhealthy: {health}"
print("Connected:", registry_url)
print("Healthy:  ", health.healthy)

## 4 · Setup helpers — UI base URL, datetime/UUID utilities, `UserCache`

A grab-bag of small helpers used throughout the rest of the script, all
short enough to live in a single cell:

- **`_ui_base_from_registry`** — derives the human-facing UI base
  (`demo.istari.app`) from the registry hostname
  (`fileservice-v2.demo.istari.app`). Used to build clickable deep links
  in the Markdown report.
- **`_iso`** — normalises a `datetime` to an ISO 8601 string and stringifies
  anything else. Every `created` field in the output payload runs through
  this so the JSON is serialisable by construction.
- **`_short`** — truncates a UUID to its first eight characters for terse
  display in the ASCII tree and the Markdown table.
- **`UserCache`** — caches `client.get_user_by_id` calls so the same user
  resolved through many lineage nodes is fetched at most once. Stores a
  stub record on lookup failure so stale or deleted users still appear in
  the digital thread instead of disappearing into an exception.

The `UserCache` demo at the end fires only if you set `USER_ID` at the top
of the notebook.


In [ ]:
# --- UI base URL ---
def _ui_base_from_registry(registry_url: str) -> str:
    """fileservice-v2.demo.istari.app -> demo.istari.app."""
    import re
    m = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", registry_url)
    if not m:
        return registry_url.rstrip("/")
    return f"{m.group(1)}{m.group(2)}"



ui_base = _ui_base_from_registry(registry_url)
print("Registry:", registry_url)
print("UI base: ", ui_base)

# --- Datetime + UUID utilities ---
def _iso(value: Any) -> str | None:
    """Convert a datetime to ISO 8601, stringify anything else."""
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.isoformat()
    return str(value)


def _short(uuid: str | None, n: int = 8) -> str:
    """Truncate a UUID to its first n characters for display."""
    return (uuid or "")[:n]



print()
print("Now:  ", _iso(datetime.now(timezone.utc)))
print("Short:", _short("a489fb15-f153-429a-bb11-5b8bae3a4a28"))

# --- UserCache ---
class UserCache:
    """Resolves user UUIDs to {id, display_name, email, ...} records, cached."""

    def __init__(self, client: Client):
        self._client = client
        self._cache: dict[str, dict[str, Any]] = {}

    def resolve(self, user_id: str | None) -> dict[str, Any] | None:
        if not user_id:
            return None
        if user_id in self._cache:
            return self._cache[user_id]
        try:
            u = self._client.get_user_by_id(user_id)
            record = {
                "id": u.id,
                "display_name": getattr(u, "display_name", None),
                "first_name": getattr(u, "first_name", None),
                "last_name": getattr(u, "last_name", None),
                "email": getattr(u, "email", None),
            }
        except Exception as exc:
            record = {"id": user_id, "display_name": None, "email": None, "_error": repr(exc)}
        self._cache[user_id] = record
        return record

    def __len__(self) -> int:
        return len(self._cache)


users = UserCache(client)

# --- UserCache demo (gated by USER_ID) ---
if USER_ID:
    print("First call (hits API):")
    print(users.resolve(USER_ID))
    print()
    print("Second call (cache hit, no API):")
    print(users.resolve(USER_ID))
    print()
    print(f"Cache holds {len(users)} user(s).")
else:
    print("(skipped — set USER_ID at the top to demo UserCache)")

## 5 · Lineage data model and Job enrichment

The dataclass + the helpers that fill it in.

**`LineageNode`** is the node type for the backward provenance tree. It
bundles a `FileRevision` (revision_id, file_id, created, ...) with the
owning Resource (resource_type, resource_id) and a classification (`step`).

**`_classify_step`** buckets a revision into one of four lifecycles:

| `step` | Meaning |
|---|---|
| `upload` | Plain file upload — no `sources` recorded |
| `job_run` | Output of a job, OR a revision whose owning resource *is* a Job |
| `promotion` | Re-uploaded as a new Model with `relationship_identifier="promoted_from"` (the standard pattern for chaining a job output into the next job) |
| `derived` | Has sources but doesn't match the above |

These four buckets are the entire vocabulary of the digital thread.

**`_job_metadata`** is the side-call we make once per Job node to enrich it
with the function/tool/agent/status fields that aren't on the bare
`FileRevision`. Every status record has its own `created_by_id` so we
resolve each through the same `UserCache` — which is how we tell apart
*who* requested a job from *who* the agent service account was that wrote
the output.

The `_job_metadata` demo at the end fires only if you set `JOB_ID`.


In [ ]:
# --- LineageNode + step classification ---
@dataclass
class LineageNode:
    """One node in a backward lineage chain (a FileRevision + its owning Resource)."""

    step: str  # upload | job_run | promotion | derived
    edge_relationship: str | None  # source.relationship_identifier (from the child)
    revision_id: str
    file_id: str | None
    resource_type: str | None  # "Model" | "Artifact" | "Job"
    resource_id: str | None
    name: str | None
    display_name: str | None
    mime: str | None
    size: int | None
    external_identifier: str | None
    created: str | None  # ISO 8601
    created_by: dict[str, Any] | None
    job: dict[str, Any] | None = None  # populated when resource_type == "Job"
    parents: list["LineageNode"] = field(default_factory=list)
    truncated: bool = False

    @property
    def label(self) -> str:
        if self.resource_type == "Job" and self.job:
            fn = self.job.get("function_name") or "job"
            return f"{fn} ({self.resource_id})"
        return self.display_name or self.name or self.revision_id


def _classify_step(rev: FileRevision, resource_type: str | None) -> str:
    if resource_type == "Job":
        return "job_run"
    sources = rev.sources or []
    if not sources:
        return "upload"
    if any(getattr(s, "relationship_identifier", None) == "promoted_from" for s in sources):
        return "promotion"
    if any(getattr(s, "resource_type", None) == "Job" for s in sources):
        return "job_run"
    return "derived"

# --- Job metadata enrichment ---
def _job_metadata(client: Client, job_id: str, users: UserCache) -> dict[str, Any]:
    """Fetch the Job and pull function/tool/status metadata for the digital thread."""
    try:
        job = client.get_job(job_id)
    except Exception as exc:
        return {"_error": f"get_job failed: {exc!r}"}

    fn = getattr(job, "function", None)
    function_block: dict[str, Any] | None = None
    if fn is not None:
        function_block = {
            "id": getattr(fn, "id", None),
            "name": getattr(fn, "name", None),
            "version": getattr(fn, "version", None),
            "module_name": getattr(fn, "module_name", None),
            "module_version": getattr(fn, "module_version", None),
            "tool_name": getattr(fn, "tool_name", None),
            "tool_display_name": getattr(fn, "tool_display_name", None),
        }

    status_history: list[dict[str, Any]] = []
    for s in (job.status_history or []):
        status_history.append({
            "id": getattr(s, "id", None),
            "name": getattr(getattr(s, "name", None), "value", None) or str(getattr(s, "name", None)),
            "message": getattr(s, "message", None),
            "created": _iso(getattr(s, "created", None)),
            "created_by": users.resolve(getattr(s, "created_by_id", None)),
        })

    return {
        "job_id": job.id,
        "function": function_block,
        "function_name": function_block["name"] if function_block else None,
        "created": _iso(getattr(job, "created", None)),
        "created_by": users.resolve(getattr(job, "created_by_id", None)),
        "assigned_agent_id": getattr(job, "assigned_agent_id", None),
        "assigned_agent_pool_id": getattr(job, "assigned_agent_pool_id", None),
        "agent_id": getattr(job, "agent_id", None),
        "status_history": status_history,
        "current_status": status_history[-1]["name"] if status_history else None,
    }

# --- _job_metadata demo (gated by JOB_ID) ---
if JOB_ID:
    meta = _job_metadata(client, JOB_ID, users)
    fn = meta.get("function") or {}
    print(f"Function: {fn.get('name')} v{fn.get('version')}")
    print(f"Module:   {fn.get('module_name')}")
    print(f"Tool:     {fn.get('tool_display_name') or fn.get('tool_name')}")
    print(f"Status:   {meta.get('current_status')}")
    print(f"Caller:   {(meta.get('created_by') or {}).get('display_name') or '?'}")
    print(f"Status events: {len(meta.get('status_history', []))}")
else:
    print("(skipped — set JOB_ID at the top to demo _job_metadata)")

## 6 · The walk and ASCII rendering

The heart of the script: the recursive walker, the two entry points that
wrap it, and the ASCII tree renderer that turns its output into something
human-readable.

### The Job-source restructuring rule

The raw platform graph is verbose: when you submit a job, the SDK uploads
a `parameters<hash>.json` blob onto the **job's own file**, and every
output artifact records *both* the input Model *and* that parameters
revision as `sources`. A naive walk shows:

```
Artifact named_cells.json
├── Model Group3-UAS-Requirements      (direct input source)
└── parameters_xxxx.json (Job)         (parameters source)
    └── Model Group3-UAS-Requirements  (same input, again — duplicated)
```

The same Model appears twice, and the meaningful node (the *Job*) is
hidden behind an opaque blob name. The rule applied below: **when a
revision has a Job source, drop the non-Job / non-`promoted_from`
siblings at that level**. The tree then reads:

```
Artifact named_cells.json
└── Job @istari:extract
    └── Model Group3-UAS-Requirements
```

Same information, fewer nodes, the Job is surfaced as the meaningful link.

### Entry points

- **`walk_single_revision(revision_id)`** — used when the user gave
  `--revision-id` or `--resource-id` (single root, no system context).
- **`walk_system(system_id)`** — walks the baseline configuration of a
  system, producing one lineage tree per tracked Model. Also returns a
  `WalkContext` carrying the v2 SDK objects needed for the system-config
  upload step later.

Both return a JSON-serialisable `payload` dict in the same shape, so the
markdown renderer and the upload step work unchanged regardless of which
entry point you used.

### ASCII rendering

`render_ascii_tree` is what you see on stdout from the CLI, and what
shows up inside fenced code blocks in the Markdown report. It walks the
JSON-shaped tree (a `dict`, not the dataclass), so it can render anything
that came out of `walk_*` or got round-tripped through JSON.

Three companion helpers feed into both the ASCII and the Markdown
renderers:

- `_count_nodes` — for the "N nodes total" stats line
- `_flatten_lineage` — DFS-flatten the tree into (depth, node) pairs;
  used by the Markdown provenance table
- `_user_label` — formats a `created_by` record. The `markdown` kwarg
  toggles between markdown-decorated and plain-ASCII output so the same
  helper feeds both renderers.

The demos at the end (1) resolve `RESOURCE_ID` to a revision via v3 if
needed and run `walk_single_revision`, then (2) run `walk_system` against
`SYSTEM_ID`, then (3) print the ASCII tree for whichever root was set up.
Each section skips gracefully when the relevant constant is empty.


In [ ]:
# --- The recursive walk ---
def build_lineage(
    client: Client,
    rev: FileRevision,
    *,
    edge_relationship: str | None,
    max_depth: int,
    depth: int,
    cache: dict[str, LineageNode],
    users: UserCache,
    source_hint: Any = None,
) -> LineageNode:
    """Recursively follow rev.sources backward to build a LineageNode tree."""
    if rev.id in cache:
        return cache[rev.id]

    # The parent's Source record usually carries resource_type/resource_id;
    # fall back to a get_file only when missing.
    resource_type = getattr(source_hint, "resource_type", None) if source_hint else None
    resource_id = getattr(source_hint, "resource_id", None) if source_hint else None
    if not resource_type and rev.file_id:
        try:
            f = client.get_file(rev.file_id)
            resource_type = getattr(f, "resource_type", None) or resource_type
            resource_id = getattr(f, "resource_id", None) or resource_id
        except Exception:
            pass

    job_block: dict[str, Any] | None = None
    if resource_type == "Job" and resource_id:
        job_block = _job_metadata(client, resource_id, users)

    node = LineageNode(
        step=_classify_step(rev, resource_type),
        edge_relationship=edge_relationship,
        revision_id=rev.id,
        file_id=rev.file_id,
        resource_type=resource_type,
        resource_id=resource_id,
        name=rev.name,
        display_name=rev.display_name,
        mime=rev.mime,
        size=rev.size,
        external_identifier=rev.external_identifier,
        created=_iso(rev.created),
        created_by=users.resolve(getattr(rev, "created_by_id", None)),
        job=job_block,
    )
    cache[rev.id] = node

    if depth >= max_depth:
        node.truncated = bool(rev.sources)
        return node

    sources = list(rev.sources or [])
    has_job_source = any(getattr(s, "resource_type", None) == "Job" for s in sources)
    if has_job_source:
        # Restructuring rule: keep only Job + promoted_from sources when a Job
        # source is present. The collapsed-out siblings reappear one level
        # deeper under the Job node, so no information is lost.
        sources = [
            s for s in sources
            if getattr(s, "resource_type", None) == "Job"
            or getattr(s, "relationship_identifier", None) == "promoted_from"
        ]

    for src in sources:
        try:
            parent_rev = client.get_revision(src.revision_id)
        except Exception:
            continue
        parent = build_lineage(
            client, parent_rev,
            edge_relationship=src.relationship_identifier,
            max_depth=max_depth,
            depth=depth + 1,
            cache=cache,
            users=users,
            source_hint=src,
        )
        node.parents.append(parent)

    return node

# --- Entry-point helpers ---
@dataclass
class WalkContext:
    """Internal context handed back from walk_system for the v2 uploader."""
    base_cfg: Any
    tracked_files: list[TrackedFile]
    existing_config_names: list[str]


def walk_single_revision(
    client: Client,
    revision_id: str,
    users: UserCache,
    max_depth: int,
) -> tuple[dict[str, Any], None]:
    """Build a JSON-ready result payload rooted at one revision."""
    rev = client.get_revision(revision_id)
    tree = build_lineage(
        client, rev,
        edge_relationship=None,
        max_depth=max_depth,
        depth=0,
        cache={},
        users=users,
    )
    if (not tree.resource_type or not tree.resource_id) and rev.file_id:
        try:
            f = client.get_file(rev.file_id)
            tree.resource_type = getattr(f, "resource_type", None) or tree.resource_type
            tree.resource_id = getattr(f, "resource_id", None) or tree.resource_id
        except Exception:
            pass

    name = tree.display_name or tree.name or f"resource {tree.resource_id or revision_id}"
    payload = {
        "system": None,
        "baseline_snapshot": None,
        "configuration": None,
        "root": {
            "revision_id": tree.revision_id,
            "resource_id": tree.resource_id,
            "resource_type": tree.resource_type,
            "name": name,
        },
        "models": [{
            "model_id": tree.resource_id,
            "model_name": name,
            "tracked_file": None,
            "lineage": asdict(tree),
        }],
    }
    return payload, None

def walk_system(
    client: Client,
    system_id: str,
    users: UserCache,
    max_depth: int,
) -> tuple[dict[str, Any], WalkContext]:
    """Walk a system's baseline configuration → one tree per tracked Model."""
    system = client.get_system(system_id)
    if not system.baseline_tagged_snapshot_id:
        raise RuntimeError(f"System {system_id} has no baseline snapshot")

    snapshot = client.get_snapshot(system.baseline_tagged_snapshot_id)
    cfg = next(
        (c for c in (system.configurations or []) if c.id == snapshot.configuration_id),
        None,
    )
    if cfg is None:
        raise RuntimeError(
            f"Snapshot {snapshot.id} points at configuration {snapshot.configuration_id}, not on system"
        )

    tracked_page = client.list_tracked_files(configuration_id=cfg.id, size=100)
    tracked = list(tracked_page.iter_items())

    models: list[dict[str, Any]] = []
    for tf in tracked:
        if not tf.resource_id:
            continue
        try:
            model = client.get_model(tf.resource_id)
        except Exception as exc:
            models.append({"model_id": tf.resource_id, "_error": repr(exc)})
            continue

        latest_rev = None
        if model.file and model.file.revisions:
            current_rev_id = tf.current_file_revision_id
            latest_rev = next(
                (r for r in model.file.revisions if r.id == current_rev_id),
                model.file.revisions[-1],
            )
        if latest_rev is None:
            models.append({"model_id": model.id, "name": getattr(model, "name", None), "_error": "no revisions"})
            continue

        m_tree = build_lineage(
            client, latest_rev,
            edge_relationship=None,
            max_depth=max_depth,
            depth=0,
            cache={},
            users=users,
        )
        # Force the root's resource_type/id to the Model that the tracked file
        # points at — otherwise the root looks like a bare Revision.
        m_tree.resource_type = "Model"
        m_tree.resource_id = model.id
        m_tree.display_name = m_tree.display_name or getattr(model, "name", None)
        models.append({
            "model_id": model.id,
            "model_name": getattr(model, "name", None),
            "tracked_file": {
                "id": tf.id,
                "specifier_type": getattr(getattr(tf, "specifier_type", None), "value", None) or str(getattr(tf, "specifier_type", None)),
                "current_file_revision_id": tf.current_file_revision_id,
                "pinned_file_revision_id": getattr(tf, "pinned_file_revision_id", None),
            },
            "lineage": asdict(m_tree),
        })

    payload = {
        "system": {
            "id": system.id,
            "name": system.name,
            "description": getattr(system, "description", None),
            "created": _iso(getattr(system, "created", None)),
            "created_by": users.resolve(getattr(system, "created_by_id", None)),
        },
        "baseline_snapshot": {
            "id": snapshot.id,
            "configuration_id": snapshot.configuration_id,
            "created": _iso(getattr(snapshot, "created", None)),
            "created_by": users.resolve(getattr(snapshot, "created_by_id", None)),
        },
        "configuration": {"id": cfg.id, "name": cfg.name},
        "models": models,
    }
    ctx = WalkContext(
        base_cfg=cfg,
        tracked_files=tracked,
        existing_config_names=[c.name for c in (system.configurations or [])],
    )
    return payload, ctx

# --- ASCII rendering helpers + render_ascii_tree ---
def _count_nodes(node: dict[str, Any] | None) -> int:
    if not node:
        return 0
    return 1 + sum(_count_nodes(p) for p in (node.get("parents") or []))


def _flatten_lineage(tree: dict[str, Any]) -> list[tuple[int, dict[str, Any]]]:
    """Return (depth, node) pairs in DFS order, deduplicating shared parents."""
    out: list[tuple[int, dict[str, Any]]] = []
    seen: set[str] = set()

    def visit(node: dict[str, Any], depth: int) -> None:
        rev_id = node.get("revision_id") or ""
        if rev_id in seen:
            return
        seen.add(rev_id)
        out.append((depth, node))
        for child in node.get("parents") or []:
            visit(child, depth + 1)

    visit(tree, 0)
    return out


def _user_label(user: dict[str, Any] | None, *, markdown: bool = True) -> str:
    if not user:
        return "_unknown_" if markdown else "unknown"
    name = user.get("display_name") or (
        f"{user.get('first_name') or ''} {user.get('last_name') or ''}".strip()
    )
    email = user.get("email")
    if name and email:
        return f"{name} <{email}>"
    if name:
        return name
    if email:
        return email
    short_id = _short(user.get("id"))
    return f"_service_ `{short_id}`" if markdown else f"service {short_id}"

def render_ascii_tree(tree: dict[str, Any]) -> str:
    """Render a lineage tree as an indented ASCII string."""
    lines: list[str] = []
    seen: set[str] = set()

    def visit(node: dict[str, Any], indent: int, is_root: bool) -> None:
        rev_id = node.get("revision_id") or ""
        already_seen = rev_id in seen
        seen.add(rev_id)

        prefix = "  " * indent
        rtype = node.get("resource_type") or "Revision"
        if rtype == "Job":
            fn = (node.get("job") or {}).get("function_name") or "job"
            rid = node.get("resource_id") or "?"
            label = f"{fn} ({rid})"
        else:
            label = (
                node.get("display_name")
                or node.get("name")
                or node.get("revision_id")
                or "?"
            )

        # Match the legacy print_tree placeholder: every non-root node
        # gets a "[via X]" suffix, and a missing relationship renders
        # as "[via -]" rather than being silently dropped.
        edge = "" if is_root else f"  [via {node.get('edge_relationship') or '-'}]"

        lines.append(f"{prefix}- {rtype} '{label}'{edge}")
        lines.append(f"{prefix}    step={node.get('step')}  rev={rev_id}")
        if node.get("resource_id"):
            lines.append(f"{prefix}    {rtype.lower()}_id={node.get('resource_id')}")
        if node.get("file_id"):
            lines.append(f"{prefix}    file_id={node.get('file_id')}")
        created = node.get("created")
        if created:
            lines.append(
                f"{prefix}    created={created}  by={_user_label(node.get('created_by'), markdown=False)}"
            )
        if node.get("job"):
            job = node["job"]
            fn = job.get("function") or {}
            lines.append(
                f"{prefix}    function={fn.get('name')} v{fn.get('version')} "
                f"module={fn.get('module_name')} tool={fn.get('tool_name')}"
            )
            status = job.get("current_status") or "?"
            agent = job.get("agent_id") or job.get("assigned_agent_id") or "-"
            lines.append(f"{prefix}    status={status}  agent={agent}")
        if node.get("truncated"):
            lines.append(f"{prefix}    ... (truncated: max_depth reached)")
        if already_seen and (node.get("parents") or []):
            lines.append(f"{prefix}    ... (subtree already shown above)")
            return
        for child in node.get("parents") or []:
            visit(child, indent + 1, False)

    visit(tree, 0, True)
    return "\n".join(lines)

# --- Demos: single-revision walk + system walk + ASCII print ---
# --- Resolve RESOURCE_ID to a revision and run a single-revision walk ---
effective_revision_id = REVISION_ID
if not effective_revision_id and RESOURCE_ID:
    resource = v3.get_resource(resource_id=RESOURCE_ID)
    effective_revision_id = resource.file_revision_id
    print(f"Resolved RESOURCE_ID {RESOURCE_ID} -> revision {effective_revision_id}")

if effective_revision_id:
    root_rev = client.get_revision(effective_revision_id)
    tree = build_lineage(
        client, root_rev,
        edge_relationship=None,
        max_depth=MAX_DEPTH,
        depth=0,
        cache={},
        users=users,
    )
    print(f"Root:    {tree.label}")
    print(f"Step:    {tree.step}")
    print(f"Parents: {len(tree.parents)}")
    print()
    print(render_ascii_tree(asdict(tree)))
else:
    tree = None
    print("(single-revision walk skipped — set REVISION_ID or RESOURCE_ID at the top)")

# --- Walk a whole system ---
if SYSTEM_ID:
    system_payload, ctx = walk_system(client, SYSTEM_ID, users, max_depth=MAX_DEPTH)
    print()
    print(f"System: {system_payload['system']['name']}")
    print(f"  baseline config: {system_payload['configuration']['name']}")
    print(f"  models tracked:  {len(system_payload['models'])}")
else:
    system_payload, ctx = None, None
    print("(system walk skipped — set SYSTEM_ID at the top)")


## 7 · Markdown report

The Markdown renderer produces a self-contained `.md` document with:

- A title block (system or root resource, baseline config, deep links)
- One section per root (model or single resource), each containing:
  - The ASCII tree inside a fenced ` ```text ` block
  - A `<details>`-collapsed provenance table with one row per node

It uses `_render_detail_row` for the table rows and `_render_model_section`
to assemble each per-root section.


In [ ]:
def _ui_link(ui_base: str, kind: str, uuid: str | None) -> str:
    if not uuid:
        return ""
    return f"{ui_base}/{kind}/{uuid}"


def _render_detail_row(node: dict[str, Any], depth: int, ui_base: str) -> str:
    rtype = node.get("resource_type") or "Revision"
    rev_id = node.get("revision_id") or ""
    short_rev = f"`{_short(rev_id)}`"

    if rtype == "Job":
        job = node.get("job") or {}
        fn = job.get("function") or {}
        fn_name = fn.get("name") or "job"
        fn_ver = fn.get("version")
        name_cell = f"{fn_name}" + (f" v{fn_ver}" if fn_ver else "")
        tool = fn.get("tool_display_name") or fn.get("tool_name")
        if tool:
            name_cell += f" _(via {tool})_"
        resource_id = node.get("resource_id")
        link = _ui_link(ui_base, "jobs", resource_id)
        id_cell = f"[`{_short(resource_id)}…`]({link})" if link else f"`{_short(resource_id)}…`"
        id_cell += f"<br/>rev {short_rev}"
    else:
        name_cell = node.get("display_name") or node.get("name") or "(unnamed)"
        resource_id = node.get("resource_id")
        kind = "models" if rtype == "Model" else "artifacts"
        link = _ui_link(ui_base, kind, resource_id)
        id_cell = f"[`{_short(resource_id)}…`]({link})" if link else f"`{_short(resource_id)}…`"
        id_cell += f"<br/>rev {short_rev}"

    edge = node.get("edge_relationship")
    edge_cell = f" _(via {edge})_" if edge and edge != "-" else ""
    indent = "&nbsp;" * (depth * 4)
    created = node.get("created") or ""
    by = _user_label(node.get("created_by"))
    return (
        f"| {indent}{node.get('step') or '?'}{edge_cell} "
        f"| {rtype} "
        f"| {name_cell} "
        f"| {id_cell} "
        f"| {created} "
        f"| {by} |"
    )

In [ ]:
def _render_model_section(m: dict[str, Any], ui_base: str) -> str:
    sect_tree = m.get("lineage") or {}
    model_id = m.get("model_id")
    model_name = m.get("model_name") or sect_tree.get("display_name") or "(unnamed)"
    tf = m.get("tracked_file") or {}
    root_type = sect_tree.get("resource_type") or "Resource"
    kind = "models" if root_type == "Model" else "resources" if root_type == "Artifact" else root_type.lower() + "s"
    type_label = "Model" if root_type == "Model" else root_type

    head = [
        f"### {model_name}",
        "",
        f"- **{type_label}:** [`{model_id}`]({_ui_link(ui_base, kind, model_id)})",
        f"- **Latest revision:** `{sect_tree.get('revision_id')}`",
        f"- **Root step:** `{sect_tree.get('step')}`",
    ]
    if tf:
        head.append(f"- **Tracked-file specifier:** {tf.get('specifier_type') or '?'}")
    head.append("")

    if "_error" in m:
        head.append(f"> Lineage error: `{m['_error']}`")
        head.append("")
        return "\n".join(head)

    diagram = "```text\n" + render_ascii_tree(sect_tree) + "\n```"

    table_header = [
        "",
        "<details>",
        "<summary>Provenance detail</summary>",
        "",
        "| Step | Type | Name | ID | Created | By |",
        "|---|---|---|---|---|---|",
    ]
    rows = [_render_detail_row(node, depth, ui_base) for depth, node in _flatten_lineage(sect_tree)]
    table_footer = ["", "</details>", ""]

    return "\n".join(head + [diagram] + table_header + rows + table_footer)


def render_markdown(result: dict[str, Any], *, ui_base: str) -> str:
    sys_meta = result.get("system") or None
    cfg_meta = result.get("configuration") or None
    snap_meta = result.get("baseline_snapshot") or {}
    root_meta = result.get("root") or None
    models = result.get("models") or []
    total_nodes = sum(_count_nodes(m.get("lineage")) for m in models)
    generated = datetime.now(timezone.utc).isoformat()

    if sys_meta:
        title = f"Digital Thread — {sys_meta.get('name')}"
    elif root_meta:
        title = f"Digital Thread — {root_meta.get('name') or root_meta.get('resource_id') or 'resource'}"
    else:
        title = "Digital Thread"

    header = [f"# {title}", ""]
    if sys_meta:
        header.append(f"- **System:** [`{sys_meta['id']}`]({_ui_link(ui_base, 'systems', sys_meta['id'])})")
        if cfg_meta:
            header.append(f"- **Baseline configuration:** {cfg_meta['name']} (`{cfg_meta['id']}`)")
        header.append(f"- **Baseline snapshot:** `{snap_meta.get('id')}`")
    elif root_meta:
        rtype = root_meta.get("resource_type") or "Resource"
        kind = "models" if rtype == "Model" else "resources" if rtype == "Artifact" else rtype.lower() + "s"
        rid = root_meta.get("resource_id")
        if rid:
            header.append(f"- **{rtype}:** [`{rid}`]({_ui_link(ui_base, kind, rid)})")
        header.append(f"- **Starting revision:** `{root_meta.get('revision_id')}`")
    header.append(f"- **Generated:** {generated}")
    header.append(f"- **Lineage:** {len(models)} root(s), {total_nodes} nodes total")
    header.append("")
    header.append("## Lineage")
    header.append("")

    sections = [_render_model_section(m, ui_base) for m in models]
    return "\n".join(header + sections) + "\n"

Render the Markdown report and display it inline. We use the
single-revision payload here; swap `single_payload` for `system_payload`
to render the system-mode report.

In [ ]:
from IPython.display import Markdown, display

if tree is not None:
    single_payload, _ = walk_single_revision(client, effective_revision_id, users, max_depth=MAX_DEPTH)
    md_text = render_markdown(single_payload, ui_base=ui_base)
    display(Markdown(md_text))
else:
    single_payload = None
    print("(skipped — need REVISION_ID or RESOURCE_ID at the top)")

## 8 · v3 upload — gated by `RUN_UPLOAD`

The first upload path uses v3 APIs end-to-end:

1. **`create_resource(path=lineage.md, resource_type=MODEL)`** —
   registers the Markdown report as a `Model` resource.
2. **`create_resource(path=lineage.json, resource_type=ARTIFACT)`** —
   registers the raw JSON payload as an `Artifact`.
3. **`create_revision_relationship`** — links the artifact revision to
   the model revision via a `produces` relationship type (the v3 SDK's
   way of saying "this Artifact was *produced by* this Model"). The
   relationship type id is fetched at runtime via
   `list_revision_relationship_types`.

The v3 resources live independently of any system — they're discoverable
via the v3 list/search endpoints and via the relationship edge. The
system-config integration is a *separate* v2 step (next section).

This cell only runs when `RUN_UPLOAD = True`.


In [ ]:
def _pick_relationship_type(v3: V3Client, preferred: str | None) -> Any:
    page = v3.list_revision_relationship_types(size=100)
    items = list(getattr(page, "items", []) or [])
    if not items:
        raise RuntimeError("No revision relationship types are defined on this registry")
    target = (preferred or "produces").lower()
    for t in items:
        if t.name.lower() == target:
            return t
    return items[0]


def upload_lineage_v3(
    v3: V3Client,
    *,
    md_path: Path,
    json_path: Path,
    system_id: str,
    system_name: str,
    relationship_type_name: str | None = None,
) -> dict[str, Any]:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    md_display = f"Digital Thread Lineage [md] — {system_name} @ {ts}"
    md_ext_id = f"digital-thread-lineage-md-{system_id}-{ts}"
    md_resource = v3.create_resource(
        path=str(md_path),
        resource_type=ResourceTypeDto.MODEL,
        display_name=md_display,
        external_identifier=md_ext_id,
        description=f"Human-readable digital thread for system {system_id} ({system_name})",
    )

    json_display = f"Digital Thread Lineage [json] — {system_name} @ {ts}"
    json_ext_id = f"digital-thread-lineage-json-{system_id}-{ts}"
    json_resource = v3.create_resource(
        path=str(json_path),
        resource_type=ResourceTypeDto.ARTIFACT,
        display_name=json_display,
        external_identifier=json_ext_id,
        description=f"Raw digital thread lineage payload for system {system_id} ({system_name})",
    )

    rel_type = _pick_relationship_type(v3, relationship_type_name)
    relationship = v3.create_revision_relationship(
        new_revision_relationship_dto=NewRevisionRelationshipDto(
            relationship_type_id=rel_type.id,
            left_revision_id=md_resource.file_revision_id,
            right_revision_id=json_resource.file_revision_id,
        )
    )
    return {
        "md_resource": {
            "resource_id": md_resource.resource_id,
            "file_id": md_resource.file_id,
            "file_revision_id": md_resource.file_revision_id,
            "display_name": md_display,
            "external_identifier": md_ext_id,
            "resource_type": ResourceTypeDto.MODEL.value,
        },
        "json_resource": {
            "resource_id": json_resource.resource_id,
            "file_id": json_resource.file_id,
            "file_revision_id": json_resource.file_revision_id,
            "display_name": json_display,
            "external_identifier": json_ext_id,
            "resource_type": ResourceTypeDto.ARTIFACT.value,
        },
        "relationship": {
            "id": relationship.id,
            "type_id": rel_type.id,
            "type_name": rel_type.name,
            "type_name_inverse": rel_type.name_inverse,
        },
    }

In [ ]:
v3_result = None
if RUN_UPLOAD and tree is not None:
    json_path = Path("walkthrough_lineage.json")
    md_path   = Path("walkthrough_lineage.md")
    json_path.write_text(json.dumps(single_payload, indent=2, default=str))
    md_path.write_text(md_text)

    scope_id = (single_payload.get("root") or {}).get("resource_id") or effective_revision_id
    scope_name = (single_payload.get("root") or {}).get("name") or "(walkthrough)"

    v3_result = upload_lineage_v3(
        v3,
        md_path=md_path,
        json_path=json_path,
        system_id=scope_id,
        system_name=scope_name,
    )
    print(f"[Model]    {v3_result['md_resource']['display_name']}")
    print(f"           resource_id={v3_result['md_resource']['resource_id']}")
    print(f"           {ui_base}/resources/{v3_result['md_resource']['resource_id']}")
    print(f"[Artifact] {v3_result['json_resource']['display_name']}")
    print(f"           resource_id={v3_result['json_resource']['resource_id']}")
    print(f"           {ui_base}/resources/{v3_result['json_resource']['resource_id']}")
    print(f"Relationship: {v3_result['relationship']['type_name']} (id={v3_result['relationship']['id']})")
else:
    print("(skipped — set RUN_UPLOAD=True and have REVISION_ID/RESOURCE_ID set)")

## 9 · v2 system-configuration upload — gated by `RUN_UPLOAD` and `SYSTEM_ID`

In system mode we also want the lineage report to appear on the system's
page in the UI. v3 resources aren't tied to systems the way v2 tracked
files are, so we make a **second** call to a v2 endpoint:

`create_configuration` with the existing tracked files preserved (LATEST /
LOCKED specifiers and pinned revisions kept exactly) plus the two new
file_ids from the v3 upload appended. The new configuration is named
`Config N — digital-thread-lineage` where N is one past the largest
`Config N` already on the system. Baseline tag is **not** moved.

Crucially: we reuse the v3-created `file_id`s in the v2 `NewTrackedFile`
entries, so the same bytes are *not* uploaded twice. The two paths share
the underlying File table.


In [ ]:
def _next_system_config_name(
    base_cfg_name: str,
    description: str,
    *,
    existing_names: list[str] | None = None,
) -> str:
    """Pick a fresh `Config N` name that doesn't collide with existing configs."""
    import re

    names = list(existing_names or [base_cfg_name])
    max_n = 0
    for nm in names:
        m = re.match(r"^Config\s+(\d+)\b", nm or "")
        if m:
            max_n = max(max_n, int(m.group(1)))
    if max_n > 0:
        return f"Config {max_n + 1} — {description}"
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%MZ")
    return f"{base_cfg_name}__{description}_{ts}"


def add_lineage_to_system_v2(
    client: Client,
    *,
    system_id: str,
    base_cfg: Any,
    tracked_files: list[TrackedFile],
    existing_config_names: list[str],
    new_file_ids: list[str],
) -> dict[str, Any]:
    """Create a new system configuration that tracks new_file_ids alongside existing ones."""
    entries: list[NewTrackedFile] = []
    for tf in tracked_files:
        if tf.specifier_type == TrackedFileSpecifierType.LOCKED:
            entries.append(NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LOCKED,
                file_id=tf.file_id,
                pinned_file_revision_id=tf.pinned_file_revision_id or tf.current_file_revision_id,
            ))
        else:
            entries.append(NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LATEST,
                file_id=tf.file_id,
            ))
    for fid in new_file_ids:
        entries.append(NewTrackedFile(
            specifier_type=TrackedFileSpecifierType.LATEST,
            file_id=fid,
        ))

    config_name = _next_system_config_name(
        base_cfg.name,
        "digital-thread-lineage",
        existing_names=existing_config_names,
    )
    new_cfg = client.create_configuration(
        system_id=system_id,
        new_system_configuration=NewSystemConfiguration(
            name=config_name,
            tracked_files=entries,
        ),
    )
    return {
        "configuration_id": new_cfg.id,
        "configuration_name": new_cfg.name,
        "tracked_count": len(entries),
        "added_file_ids": list(new_file_ids),
    }

In [ ]:
if RUN_UPLOAD and SYSTEM_ID and ctx is not None and system_payload is not None:
    # The system-mode report is a different file from the single-revision one.
    sys_md_text = render_markdown(system_payload, ui_base=ui_base)
    sys_json_path = Path("walkthrough_system_lineage.json")
    sys_md_path   = Path("walkthrough_system_lineage.md")
    sys_json_path.write_text(json.dumps(system_payload, indent=2, default=str))
    sys_md_path.write_text(sys_md_text)

    # 1) v3 upload to register both files and get their file_ids
    sys_v3 = upload_lineage_v3(
        v3,
        md_path=sys_md_path,
        json_path=sys_json_path,
        system_id=system_payload["system"]["id"],
        system_name=system_payload["system"]["name"],
    )
    # 2) v2 step: add those file_ids to a new system configuration
    v2_result = add_lineage_to_system_v2(
        client,
        system_id=system_payload["system"]["id"],
        base_cfg=ctx.base_cfg,
        tracked_files=ctx.tracked_files,
        existing_config_names=ctx.existing_config_names,
        new_file_ids=[sys_v3["md_resource"]["file_id"], sys_v3["json_resource"]["file_id"]],
    )
    print(f"New configuration: {v2_result['configuration_name']} ({v2_result['configuration_id']})")
    print(f"Tracked files:     {v2_result['tracked_count']} ({len(v2_result['added_file_ids'])} new)")
    print(f"View on system:    {ui_base}/systems/{system_payload['system']['id']}")
else:
    print("(skipped — needs RUN_UPLOAD=True, SYSTEM_ID set, and walk_system run earlier)")

## 10 · Wrap-up

You just reconstructed the entirety of `build_lineage.py` block by block.
Recap of what's now in scope:

- A working v2 `Client` + v3 `V3Client`, both pointing at the registry
  in your env vars.
- A `UserCache` that caches user lookups across the whole notebook.
- `LineageNode` + `build_lineage` — the recursive walk with the
  Job-source restructuring rule baked in.
- Two entry points (`walk_single_revision`, `walk_system`) producing a
  shared JSON payload shape.
- `render_ascii_tree` and `render_markdown` for the two output formats.
- v3 and v2 upload paths, both gated behind `RUN_UPLOAD`.

To use any of this in your own work, you can either:

1. **Copy the functions** out of this notebook into your own script.
2. **Import from `build_lineage.py`** in the same directory:
   `from build_lineage import build_lineage, render_markdown, ...`.
3. **Run the CLI directly**: `python build_lineage.py <system_id> --upload`
   or `python build_lineage.py --resource-id <uuid> --upload`.

Where to go next:

- Run `run_chain.py` (also in this directory) to seed a fresh 6-deep
  promotion chain, then plug the resulting `final_named_cells_resource_id`
  into `RESOURCE_ID` at the top of this notebook and re-run.
- Tweak `_classify_step` or the Job-restructuring rule and watch the
  tree change shape — the recursion primitive is general-purpose.
- Replace `render_ascii_tree` with your own renderer (DOT graph, JSON-LD,
  whatever fits your downstream tool) — `walk_*` outputs a stable
  JSON-serializable shape that's easy to retarget.
